In [1]:
!pip install roboflow
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.5/84.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 95.2 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.11.0.86
    Uninstalling opencv-python-headless-4.11.0.86:
      Successfully uninstalled opencv-python-headless-4.11.0.86
  Attempting uninstall: idna
    Found existing installation: idna 3.10
    Uninstalling idna-3.10:
      Successfully uninstalled idna-3.10
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 949.8/949.8 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 106.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 82.2 MB/s eta 0:00:00

In [10]:
import os
from roboflow import Roboflow
from google.colab import userdata
from ultralytics import YOLO
from ultralytics import RTDETR
import matplotlib.pyplot as plt
import cv2
import numpy as np
import shutil

#for UTF8 error when downloading ultralytics results from colab
import locale
locale.getpreferredencoding = lambda: "UTF-8"

#google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
def visualize_results(test_results, result_show_limit=20, txt_save_file=None):
  grid_imgs = []
  img_box_counts = []
  img_names = []
  for i, result in enumerate(test_results):
      if i >= result_show_limit:
          break
      # result.plot() returns an image (NumPy array) with predictions overlaid
      img = result.plot()
      grid_imgs.append(img)
      img_box_counts.append(len(result.boxes))
      img_names.append(result.path)

  cols = 4
  rows = int(np.ceil(len(grid_imgs) / cols))

  fig, axs = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))

  for idx, ax in enumerate(axs.flat):
      if idx < len(grid_imgs):

          ax.imshow(grid_imgs[idx])
          ax.set_title(f"Result {idx+1} with {img_box_counts[idx]} staves detected")
      ax.axis('off')

  if txt_save_file is not None:
      with open(txt_save_file, 'w') as f:
          for idx, img_name in enumerate(img_names):
              f.write(f"{img_box_counts[idx]} staves detected in image: {img_name}\n")

  plt.tight_layout()
  plt.show()

def save_results_to_indvidual_txt_files(test_results, output_folder, staves_threshold=500):
  os.makedirs(output_folder, exist_ok=True)
  os.makedirs(os.path.join(output_folder, "labels"), exist_ok=True)
  os.makedirs(os.path.join(output_folder, "images"), exist_ok=True)

  for i, result in enumerate(test_results):
    img_path = result.path
    img_name = os.path.basename(img_path)
    img_name_without_ext = os.path.splitext(img_name)[0]
    txt_file_path = os.path.join(output_folder, 'labels', f"{img_name_without_ext}.txt")

    conf_list = result.boxes.conf.cpu().numpy()
    box_list = result.boxes.xywhn
    cls_list = result.boxes.cls.cpu().numpy()

    box_count = len(conf_list)
    if box_count>=staves_threshold:
      shutil.copy(img_path, os.path.join(output_folder, "images", img_name))
      with open(txt_file_path, 'a') as result_file:
        for conf, box, cls in zip(conf_list, box_list, cls_list):
          if conf>0.1:
            result_file.write(f"{int(cls)} {box[0]} {box[1]} {box[2]} {box[3]}\n")

In [5]:
weights_path = "/content/drive/MyDrive/Stave_Project/Developers/modelWeights/runs/detect/train2/weights/best.pt"
input_folder = "/content/drive/MyDrive/Stave_Project/Developers/staveProjectImgsAllCropped"
output_folder = "/content/drive/MyDrive/Stave_Project/Developers/staveProjectImgsAllCroppedResults"

In [6]:
model = YOLO(weights_path)
test_results = []
for result in model.predict(source=input_folder, half=True, stream=True, max_det=1000):
  test_results.append(result)


image 1/696 /content/drive/MyDrive/Stave_Project/Developers/staveProjectImgsAllCropped/17289132322424750858174835555844 - Brandon Wagner_0.jpeg: 320x512 703 woods, 69.6ms
image 2/696 /content/drive/MyDrive/Stave_Project/Developers/staveProjectImgsAllCropped/17289134066225452634665236915044 - Brandon Wagner_0.jpeg: 320x512 688 woods, 10.9ms
image 3/696 /content/drive/MyDrive/Stave_Project/Developers/staveProjectImgsAllCropped/17289178627772986676454615492992 - Brandon Wagner_0.jpeg: 320x512 494 woods, 10.9ms
image 4/696 /content/drive/MyDrive/Stave_Project/Developers/staveProjectImgsAllCropped/1728932532448232548761055507990 - Brandon Wagner_0.jpeg: 288x512 366 woods, 69.0ms
image 5/696 /content/drive/MyDrive/Stave_Project/Developers/staveProjectImgsAllCropped/17289325762097634202431665002761 - Brandon Wagner_0.jpeg: 320x512 537 woods, 11.6ms
image 6/696 /content/drive/MyDrive/Stave_Project/Developers/staveProjectImgsAllCropped/1728932752409444287612924354671 - Brandon Wagner_0.jpeg: 2

In [18]:
save_results_to_indvidual_txt_files(test_results, output_folder)